# FitRec no Kaggle (com continuação depois de 12 h)

O Kaggle encerra uma sessão em 12 h. Com `--max-hours`, o treino para sozinho antes disso, no fim de uma época, com o checkpoint (`last.pt`) salvo nos outputs. Depois é só rodar uma nova versão que continua dali.

**Primeira vez**
1. *Add Input* → o dataset com o `fitrec_compact.npz`.
2. *Settings* → *Accelerator: GPU (T4 ou P100)* e *Internet: On* (para o `git clone`).
3. Na célula de configuração, deixe `CONTINUAR = False` e escolha o `EXPERIMENTO`.
4. *Save Version* → *Save & Run All (Commit)*. Roda em segundo plano (pode fechar a aba) e os outputs ficam salvos na versão.

**Para continuar** (quando o log terminar com `time limit: ... stopped after epoch N`)
1. *Add Input* → *Notebook Output Files* → este notebook (a versão que acabou de rodar).
2. Mude `CONTINUAR = True`.
3. *Save Version* → *Save & Run All* de novo. Ele copia a execução anterior e continua da última época concluída, com as mesmas opções. Repita até o log mostrar `test loss`.


In [ ]:
# ---- configuração ----
BRANCH = "mamba"
EXPERIMENTO = "--arch mamba --lr-decay 0.7 --min-delta 0.01"  # só vale com CONTINUAR = False
MAX_HORAS = 11      # o Kaggle encerra em 12 h; o treino para antes da época que passaria disso
CONTINUAR = False   # True = continua a execução que está nos inputs (output da versão anterior)
EXECUCAO = ""       # só se houver mais de uma execução nos inputs: o nome da pasta

In [ ]:
import glob, os, shutil

CODE = "/tmp/fit-rec/FitRec-Attn"   # código fora de /kaggle/working, para não ir para os outputs
OUT = "/kaggle/working/fitrec-attn" # checkpoints: estes vão para os outputs
!rm -rf /tmp/fit-rec && git clone -q -b {BRANCH} https://github.com/laracmv/fit-rec.git /tmp/fit-rec

npz = glob.glob("/kaggle/input/**/fitrec_compact.npz", recursive=True)
assert npz, "adicione como input o dataset com o fitrec_compact.npz"
os.makedirs(CODE + "/data", exist_ok=True)
os.makedirs(OUT + "/model_states", exist_ok=True)
os.symlink(npz[0], CODE + "/data/fitrec_compact.npz")
os.symlink(OUT, CODE + "/data/fitrec-attn")
print("dados:", npz[0])

!pip install -q haversine
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
run_arg = EXPERIMENTO
if CONTINUAR:
    found = sorted(glob.glob("/kaggle/input/**/fitrec-attn/model_states/*/last.pt", recursive=True))
    runs = {os.path.basename(os.path.dirname(p)): os.path.dirname(p) for p in found}
    assert runs, "nenhum last.pt nos inputs: adicione o output da versão anterior deste notebook"
    if EXECUCAO:
        name = EXECUCAO
    else:
        assert len(runs) == 1, "mais de uma execução nos inputs, escolha uma em EXECUCAO: {}".format(list(runs))
        name = next(iter(runs))
    shutil.copytree(runs[name], OUT + "/model_states/" + name, dirs_exist_ok=True)
    for log in glob.glob("/kaggle/input/**/treino.log", recursive=True)[:1]:
        shutil.copy(log, "/kaggle/working/treino.log")  # o log continua no mesmo arquivo
    run_arg = "--resume data/fitrec-attn/model_states/" + name
    print("continuando:", name)
print("python fitrect-attn.py", run_arg, "--max-hours", MAX_HORAS)

In [ ]:
%cd {CODE}
!python -u fitrect-attn.py {run_arg} --max-hours {MAX_HORAS} 2>&1 | tee -a /kaggle/working/treino.log